# AIT511 Machine Learning Project I - Vehicle Type Classification

**Team:** DAPS (Shailesh, Anik, Dev, Praveenraj) | **TA:** Akshat Batra

## Course Topics Covered
- Bayes Optimal Classifier (Quadratic - QDA)
- Bayes Optimal Classifier (Linear - LDA)
- Naive Bayes Classifier (Quadratic)
- Naive Bayes Classifier (Linear)
- Binary Logistic Regression (Gradient Descent & BCE)
- Multiclass Logistic Regression (Softmax)
- Classification Trees with & without Pruning
- EDA, Data Splitting, Cross-Validation
- Regularization
- Gradient Descent Methods (Stochastic, Full, Batch)
- Testing & Demonstrating Different Error Types

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
from sklearn.base import BaseEstimator, ClassifierMixin
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Exploratory Data Analysis (EDA)

In [ ]:
train = pd.read_csv('vehicle/train.csv')
test = pd.read_csv('vehicle/test.csv')
sample_sub = pd.read_csv('vehicle/sample_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")
print(f"\nTrain columns: {list(train.columns)}")
print(f"Test columns: {list(test.columns)}")
print(f"\nMissing values:\n{train.isnull().sum()}")
print(f"\nClass distribution:\n{train['vehicle_class'].value_counts().sort_index()}")

In [ ]:
# Feature distributions by class
features = [c for c in train.columns if c not in ['vehicle_id', 'vehicle_class']]

fig, axes = plt.subplots(5, 4, figsize=(16, 18))
axes = axes.flatten()

for i, feat in enumerate(features):
    if i < len(axes):
        for cls in [0, 1, 2]:
            data = train[train['vehicle_class'] == cls][feat]
            axes[i].hist(data, bins=30, alpha=0.5, label=f'Class {cls}', density=True)
        axes[i].set_title(feat)
        axes[i].legend(fontsize=8)
        axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_matrix = train[features].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot for key features
key_features = ['horsepower_hp', 'vehicle_weight_kg', 'engine_displacement_l', 
                'acceleration_0_100_kmh_s', 'torque_nm', 'drag_coefficient']

sns.pairplot(train[key_features + ['vehicle_class']], hue='vehicle_class', 
             palette='Set2', diag_kind='hist', plot_kws={'alpha': 0.6, 's': 20})
plt.suptitle('Key Feature Pairplot by Vehicle Class', y=1.02)
plt.show()

## 2. Data Preprocessing

In [ ]:
# Check for duplicates
print(f"Duplicate rows in train: {train.duplicated().sum()}")
print(f"Duplicate vehicle_ids in train: {train['vehicle_id'].duplicated().sum()}")
print(f"Duplicate rows in test: {test.duplicated().sum()}")

# Remove vehicle_id (non-predictive)
X = train.drop(['vehicle_id', 'vehicle_class'], axis=1)
y = train['vehicle_class'].values
X_test = test.drop(['vehicle_id'], axis=1)

# Train/Validation split (stratified)
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_tr.shape}, Val: {X_val.shape}")
print(f"Classes: {np.unique(y_tr, return_counts=True)}")

## 3. Bayes Optimal Classifiers

### 3.1 Quadratic Discriminant Analysis (QDA) - Bayes Optimal Quadratic Classifier

In [ ]:
# QDA - assumes different covariance per class (quadratic boundary)
qda = QuadraticDiscriminantAnalysis()
qda.fit(X_tr_scaled, y_tr)
y_pred_qda = qda.predict(X_val_scaled)

acc_qda = accuracy_score(y_val, y_pred_qda)
f1_qda = f1_score(y_val, y_pred_qda, average='macro')

print(f"QDA (Bayes Optimal Quadratic) Results:")
print(f"  Accuracy: {acc_qda:.4f}")
print(f"  Macro F1: {f1_qda:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_qda))

### 3.2 Linear Discriminant Analysis (LDA) - Bayes Optimal Linear Classifier

In [ ]:
# LDA - assumes shared covariance per class (linear boundary)
lda = LinearDiscriminantAnalysis()
lda.fit(X_tr_scaled, y_tr)
y_pred_lda = lda.predict(X_val_scaled)

acc_lda = accuracy_score(y_val, y_pred_lda)
f1_lda = f1_score(y_val, y_pred_lda, average='macro')

print(f"LDA (Bayes Optimal Linear) Results:")
print(f"  Accuracy: {acc_lda:.4f}")
print(f"  Macro F1: {f1_lda:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_lda))

### 3.3 Naive Bayes (Quadratic) - Diagonal Covariance

In [ ]:
# Gaussian Naive Bayes - diagonal covariance (quadratic boundary due to different variances)
gnb = GaussianNB()
gnb.fit(X_tr_scaled, y_tr)
y_pred_nb = gnb.predict(X_val_scaled)

acc_nb = accuracy_score(y_val, y_pred_nb)
f1_nb = f1_score(y_val, y_pred_nb, average='macro')

print(f"Naive Bayes (Quadratic) Results:")
print(f"  Accuracy: {acc_nb:.4f}")
print(f"  Macro F1: {f1_nb:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_nb))

### 3.4 Naive Bayes (Linear) - Shared Variance

In [ ]:
# Naive Bayes with shared variance (linear boundary)
# This is equivalent to LDA with diagonal covariance
class NaiveBayesLinear:
    """Naive Bayes with shared diagonal covariance (linear decision boundary)"""
    def __init__(self):
        self.classes = None
        self.priors = None
        self.means = None
        self.vars = None  # shared across classes
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_features = X.shape[1]
        
        self.means = np.zeros((n_classes, n_features))
        self.priors = np.zeros(n_classes)
        
        # Compute class means and priors
        for i, cls in enumerate(self.classes):
            X_cls = X[y == cls]
            self.means[i] = X_cls.mean(axis=0)
            self.priors[i] = len(X_cls) / len(X)
        
        # Shared diagonal covariance (pooled variance)
        self.vars = np.zeros(n_features)
        for i, cls in enumerate(self.classes):
            X_cls = X[y == cls]
            self.vars += X_cls.var(axis=0) * (len(X_cls) - 1)
        self.vars /= (len(X) - n_classes)
        # Avoid zero variance
        self.vars = np.maximum(self.vars, 1e-9)
        return self
    
    def predict(self, X):
        n_samples = X.shape[0]
        log_posteriors = np.zeros((n_samples, len(self.classes)))
        
        for i, cls in enumerate(self.classes):
            # Log prior
            log_post = np.log(self.priors[i])
            # Log likelihood (Gaussian with diagonal covariance)
            diff = X - self.means[i]
            log_post += -0.5 * np.sum(np.log(2 * np.pi * self.vars))
            log_post += -0.5 * np.sum(diff**2 / self.vars, axis=1)
            log_posteriors[:, i] = log_post
        
        return self.classes[np.argmax(log_posteriors, axis=1)]

nbl = NaiveBayesLinear()
nbl.fit(X_tr_scaled, y_tr)
y_pred_nbl = nbl.predict(X_val_scaled)

acc_nbl = accuracy_score(y_val, y_pred_nbl)
f1_nbl = f1_score(y_val, y_pred_nbl, average='macro')

print(f"Naive Bayes (Linear/Shared Variance) Results:")
print(f"  Accuracy: {acc_nbl:.4f}")
print(f"  Macro F1: {f1_nbl:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_nbl))

## 4. Logistic Regression

### 4.1 Binary Logistic Regression from Scratch (Gradient Descent + BCE)

In [ ]:
class LogisticRegressionGD:
    """Binary Logistic Regression with Gradient Descent (One-vs-Rest for multiclass)"""
    def __init__(self, lr=0.1, epochs=1000, batch_size=None, verbose=False, reg_lambda=0.0):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.reg_lambda = reg_lambda  # L2 regularization
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def sigmoid(self, z):
        z = np.clip(z, -500, 500)  # Prevent overflow
        return 1 / (1 + np.exp(-z))
    
    def fit_binary(self, X, y):
        """Fit for binary classification {0, 1}"""
        n_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        if self.batch_size is None:
            self.batch_size = n_samples
        
        for epoch in range(self.epochs):
            if self.batch_size < n_samples:
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                y_shuffled = y[indices]
            else:
                X_shuffled = X
                y_shuffled = y
            
            epoch_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                y_batch = y_shuffled[i:i+self.batch_size]
                
                # Forward pass
                z = X_batch @ self.weights + self.bias
                y_pred = self.sigmoid(z)
                
                # Binary Cross-Entropy Loss
                eps = 1e-15
                y_pred_clipped = np.clip(y_pred, eps, 1 - eps)
                loss = -np.mean(y_batch * np.log(y_pred_clipped) + 
                               (1 - y_batch) * np.log(1 - y_pred_clipped))
                
                # L2 regularization
                if self.reg_lambda > 0:
                    loss += 0.5 * self.reg_lambda * np.sum(self.weights**2)
                
                # Gradients
                dz = y_pred - y_batch
                dw = (1 / len(y_batch)) * X_batch.T @ dz + self.reg_lambda * self.weights
                db = (1 / len(y_batch)) * np.sum(dz)
                
                # Update
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
                epoch_loss += loss
                n_batches += 1
            
            avg_loss = epoch_loss / n_batches
            self.loss_history.append(avg_loss)
            
            if self.verbose and epoch % 100 == 0:
                print(f"  Epoch {epoch}: Loss = {avg_loss:.4f}")
        
        return self
    
    def predict_proba_binary(self, X):
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def predict_binary(self, X):
        return (self.predict_proba_binary(X) > 0.5).astype(int)
    
    def fit(self, X, y):
        """One-vs-Rest for multiclass"""
        self.classes = np.unique(y)
        self.n_classes = len(self.classes)
        self.binary_classifiers = []
        
        for i, cls in enumerate(self.classes):
            if self.verbose:
                print(f"Training OvR classifier for class {cls} vs rest...")
            y_binary = (y == cls).astype(int)
            clf = LogisticRegressionGD(
                lr=self.lr, epochs=self.epochs, 
                batch_size=self.batch_size, verbose=self.verbose,
                reg_lambda=self.reg_lambda
            )
            clf.fit_binary(X, y_binary)
            self.binary_classifiers.append(clf)
        return self
    
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], self.n_classes))
        for i, clf in enumerate(self.binary_classifiers):
            probas[:, i] = clf.predict_proba_binary(X)
        # Normalize to sum to 1
        probas = probas / probas.sum(axis=1, keepdims=True)
        return probas
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes[np.argmax(probas, axis=1)]

In [ ]:
# Train custom Logistic Regression (OvR)
print("=== Custom Logistic Regression (OvR, Full Batch GD) ===")
lr_custom = LogisticRegressionGD(lr=0.1, epochs=500, batch_size=None, verbose=True, reg_lambda=0.01)
lr_custom.fit(X_tr_scaled, y_tr)
y_pred_lr_custom = lr_custom.predict(X_val_scaled)

acc_lr_custom = accuracy_score(y_val, y_pred_lr_custom)
f1_lr_custom = f1_score(y_val, y_pred_lr_custom, average='macro')

print(f"Custom Logistic Regression (OvR) Results:")
print(f"  Accuracy: {acc_lr_custom:.4f}")
print(f"  Macro F1: {f1_lr_custom:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_lr_custom))

In [ ]:
# Plot loss curves for each binary classifier
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (clf, cls) in enumerate(zip(lr_custom.binary_classifiers, lr_custom.classes)):
    axes[i].plot(clf.loss_history)
    axes[i].set_title(f'Class {cls} vs Rest (BCE Loss)')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')

plt.tight_layout()
plt.show()

### 4.2 Multinomial Logistic Regression (Softmax) from Scratch

In [ ]:
class MultinomialLogisticRegression:
    """Multinomial Logistic Regression (Softmax) with Gradient Descent"""
    def __init__(self, lr=0.1, epochs=1000, batch_size=None, verbose=False, reg_lambda=0.0):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose
        self.reg_lambda = reg_lambda
        self.weights = None  # shape: (n_features, n_classes)
        self.bias = None     # shape: (n_classes,)
        self.loss_history = []
    
    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)  # Numerical stability
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # One-hot encode labels
        Y = np.zeros((n_samples, n_classes))
        for i, cls in enumerate(self.classes):
            Y[y == cls, i] = 1
        
        # Initialize weights
        self.weights = np.random.randn(n_features, n_classes) * 0.01
        self.bias = np.zeros(n_classes)
        
        if self.batch_size is None:
            self.batch_size = n_samples
        
        for epoch in range(self.epochs):
            if self.batch_size < n_samples:
                indices = np.random.permutation(n_samples)
                X_shuffled = X[indices]
                Y_shuffled = Y[indices]
            else:
                X_shuffled = X
                Y_shuffled = Y
            
            epoch_loss = 0
            n_batches = 0
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i+self.batch_size]
                Y_batch = Y_shuffled[i:i+self.batch_size]
                
                # Forward pass
                logits = X_batch @ self.weights + self.bias
                probas = self.softmax(logits)
                
                # Categorical Cross-Entropy Loss
                eps = 1e-15
                probas_clipped = np.clip(probas, eps, 1 - eps)
                loss = -np.mean(np.sum(Y_batch * np.log(probas_clipped), axis=1))
                
                # L2 regularization
                if self.reg_lambda > 0:
                    loss += 0.5 * self.reg_lambda * np.sum(self.weights**2)
                
                # Gradients
                dlogits = probas - Y_batch
                dw = (1 / len(Y_batch)) * X_batch.T @ dlogits + self.reg_lambda * self.weights
                db = (1 / len(Y_batch)) * np.sum(dlogits, axis=0)
                
                # Update
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
                epoch_loss += loss
                n_batches += 1
            
            avg_loss = epoch_loss / n_batches
            self.loss_history.append(avg_loss)
            
            if self.verbose and epoch % 100 == 0:
                print(f"  Epoch {epoch}: Loss = {avg_loss:.4f}")
        
        return self
    
    def predict_proba(self, X):
        logits = X @ self.weights + self.bias
        return self.softmax(logits)
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return self.classes[np.argmax(probas, axis=1)]

In [ ]:
# Train Multinomial Logistic Regression
print("=== Multinomial Logistic Regression (Softmax, Full Batch GD) ===")
mlr = MultinomialLogisticRegression(lr=0.1, epochs=500, batch_size=None, verbose=True, reg_lambda=0.01)
mlr.fit(X_tr_scaled, y_tr)
y_pred_mlr = mlr.predict(X_val_scaled)

acc_mlr = accuracy_score(y_val, y_pred_mlr)
f1_mlr = f1_score(y_val, y_pred_mlr, average='macro')

print(f"Multinomial Logistic Regression (Softmax) Results:")
print(f"  Accuracy: {acc_mlr:.4f}")
print(f"  Macro F1: {f1_mlr:.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_mlr))

In [ ]:
# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(mlr.loss_history)
plt.title('Multinomial Logistic Regression (Categorical Cross-Entropy Loss)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

### 4.3 Sklearn Logistic Regression with Different GD Variants

In [ ]:
# Compare sklearn logistic regression with different solvers
solvers = [
    ('lbfgs', 'Full Batch (L-BFGS)'),
    ('saga', 'SAGA (supports SGD/minibatch)'),
    ('sag', 'SAG (Stochastic Average Gradient)'),
]

for solver, name in solvers:
    lr = LogisticRegression(solver=solver, max_iter=1000, C=100, 
                            multi_class='multinomial', random_state=42)
    lr.fit(X_tr_scaled, y_tr)
    y_pred = lr.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    print(f"{name:30s}: Accuracy={acc:.4f}, Macro F1={f1:.4f}")

## 5. SGD Classifier (Stochastic Gradient Descent)

In [ ]:
# SGDClassifier with different loss functions
losses = [('log_loss', 'Logistic (SGD)'), 
          ('hinge', 'Linear SVM (SGD)'),
          ('modified_huber', 'Modified Huber (SGD)')]

for loss, name in losses:
    sgd = SGDClassifier(loss=loss, max_iter=1000, random_state=42,
                        learning_rate='optimal', alpha=0.0001)
    sgd.fit(X_tr_scaled, y_tr)
    y_pred = sgd.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    print(f"{name:30s}: Accuracy={acc:.4f}, Macro F1={f1:.4f}")

## 6. Classification Trees with Pruning

In [ ]:
# Decision Tree without pruning
dt_no_prune = DecisionTreeClassifier(random_state=42)
dt_no_prune.fit(X_tr_scaled, y_tr)
y_pred_dt = dt_no_prune.predict(X_val_scaled)
acc_dt = accuracy_score(y_val, y_pred_dt)
print(f"Decision Tree (no pruning) Accuracy: {acc_dt:.4f}")
print(f"Tree depth: {dt_no_prune.get_depth()}")
print(f"Number of leaves: {dt_no_prune.get_n_leaves()}")

# Pre-pruning with max_depth
for max_depth in [3, 5, 7, 10, 15, 20]:
    dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    dt.fit(X_tr_scaled, y_tr)
    y_pred = dt.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    print(f"max_depth={max_depth}: Acc={acc:.4f}, depth={dt.get_depth()}, leaves={dt.get_n_leaves()}")

In [ ]:
# Cost-Complexity Pruning (post-pruning)
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_tr_scaled, y_tr)

path = dt_full.cost_complexity_pruning_path(X_tr_scaled, y_tr)
ccp_alphas = path.ccp_alphas

print(f"Number of ccp_alphas: {len(ccp_alphas)}")

# Evaluate each alpha on validation set
acc_pruned = []
for alpha in ccp_alphas[:-1]:
    dt_pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    dt_pruned.fit(X_tr_scaled, y_tr)
    y_pred = dt_pruned.predict(X_val_scaled)
    acc_pruned.append(accuracy_score(y_val, y_pred))

best_idx = np.argmax(acc_pruned)
best_alpha = ccp_alphas[best_idx]
print(f"\nBest ccp_alpha: {best_alpha:.6f}")
print(f"Best Accuracy: {acc_pruned[best_idx]:.4f}")

In [ ]:
# Train final pruned tree
dt_best = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
dt_best.fit(X_tr_scaled, y_tr)
y_pred_best = dt_best.predict(X_val_scaled)
acc_best = accuracy_score(y_val, y_pred_best)

print(f"Pruned Tree: Acc={acc_best:.4f}, depth={dt_best.get_depth()}, leaves={dt_best.get_n_leaves()}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_best))

## 7. Cross-Validation & Model Comparison

In [ ]:
# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_cv = {
    'QDA (Bayes Quadratic)': QuadraticDiscriminantAnalysis(),
    'LDA (Bayes Linear)': LinearDiscriminantAnalysis(),
    'Naive Bayes (Quadratic)': GaussianNB(),
    'Naive Bayes (Linear)': NaiveBayesLinear(),
    'Logistic Regression (OvR)': LogisticRegression(max_iter=1000, C=100),
    'Multinomial LR (Softmax)': LogisticRegression(max_iter=1000, C=100, multi_class='multinomial'),
    'Decision Tree (pruned)': DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SGD (Log Loss)': SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
}

cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X_tr_scaled, y_tr, cv=skf, scoring='accuracy')
    cv_results[name] = (scores.mean(), scores.std())
    print(f"{name:35s}: Acc = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Detailed classification report for best model
best_model_name = max(cv_results, key=lambda k: cv_results[k][0])
best_model = models_cv[best_model_name]
best_model.fit(X_tr_scaled, y_tr)
y_pred_best = best_model.predict(X_val_scaled)

print(f"Best model: {best_model_name}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred_best, target_names=['Sedan (0)', 'SUV (1)', 'Sports Car (2)']))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_val, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Sedan (0)', 'SUV (1)', 'Sports Car (2)'],
            yticklabels=['Sedan (0)', 'SUV (1)', 'Sports Car (2)'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 8. Ensemble Methods

In [ ]:
# Stacking Ensemble
estimators = [
    ('qda', QuadraticDiscriminantAnalysis()),
    ('lda', LinearDiscriminantAnalysis()),
    ('nb', GaussianNB()),
    ('lr', LogisticRegression(max_iter=1000, C=100, multi_class='multinomial')),
    ('dt', DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42)),
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, C=10),
    cv=5
)

stacking.fit(X_tr_scaled, y_tr)
y_pred_stack = stacking.predict(X_val_scaled)
acc_stack = accuracy_score(y_val, y_pred_stack)
f1_stack = f1_score(y_val, y_pred_stack, average='macro')

print(f"Stacking Ensemble: Accuracy={acc_stack:.4f}, Macro F1={f1_stack:.4f}")

## 9. Generate Submission

In [ ]:
# Retrain on full training data
stacking.fit(X, y)

# Predict on test set
test_pred = stacking.predict(X_test_scaled)

# Create submission
submission = pd.DataFrame({
    'vehicle_id': test['vehicle_id'],
    'vehicle_class': test_pred
})

print(f"Submission shape: {submission.shape}")
print(submission.head())
print(f"\nClass distribution in submission:\n{submission['vehicle_class'].value_counts().sort_index()}")

submission.to_csv('vehicle/submission.csv', index=False)
print("Saved to vehicle/submission.csv")

In [ ]:
# Verify submission format
sample = pd.read_csv('vehicle/sample_submission.csv')
print(f"Sample submission shape: {sample.shape}")
print(f"Our submission shape: {submission.shape}")
print(f"Columns match: {list(sample.columns) == list(submission.columns)}")
print(f"ID match: {(sample['vehicle_id'] == submission['vehicle_id']).all()}")